#  Fake News Detection System

## 1. Environment Setup

In [2]:

import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
import os

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load and Prepare LIAR Dataset
The LIAR dataset is more complex than ISOT, labeled with 6 degrees of truth. We will map them to Binary (True/Fake) for compatibility.

In [ ]:
def load_liar_dataset(path):
    col_names = ['id', 'label', 'statement', 'subject', 'speaker', 'job', 'state', 'party', 'barely_true_c', 'false_c', 'half_true_c', 'mostly_true_c', 'pants_on_fire_c', 'context']
    try:
        df = pd.read_csv(path, sep='\t', header=None, names=col_names, quoting=3) # quoting=3 (QUOTE_NONE) helps with parsing errors
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return pd.DataFrame() 
    return df[['label', 'statement']]

def map_liar_labels(label):
   
    if label in ['pants-fire', 'false', 'barely-true']:
        return 1 # Fake
    elif label in ['true', 'mostly-true']:
        return 0 # Real
    else:
        return -1 

try:
    train_df = load_liar_dataset('../Data/LiarData/train.tsv')
    test_df = load_liar_dataset('../Data/LiarData/test.tsv')
    valid_df = load_liar_dataset('../Data/LiarData/valid.tsv')

    liar_df = pd.concat([train_df, test_df, valid_df])
    
    if not liar_df.empty:
        liar_df['binary_label'] = liar_df['label'].apply(map_liar_labels)
        
        liar_df = liar_df[liar_df['binary_label'] != -1].reset_index(drop=True)
        
        print(f"LIAR Dataset Loaded: {len(liar_df)} rows")
        print(liar_df['binary_label'].value_counts())
    else:
        print("LIAR Dataset loaded but empty or failed.")
    
except FileNotFoundError:
    print("LIAR dataset not found. Please ensure files are in ../Data/LiarData/")

LIAR Dataset Loaded: 10198 rows
binary_label
1    5669
0    4529
Name: count, dtype: int64


## 3. Prepare ISOT Dataset (for Training)
We will train on ISOT (large, clean) and test on LIAR (complex, unseen) to check robustness.

In [ ]:
# Load ISOT
fake_df = pd.read_csv('../Data/Fake.csv')
true_df = pd.read_csv('../Data/True.csv')

fake_df['label'] = 1
true_df['label'] = 0
import re
def clean_text(text):

    text = re.sub(r"^.*?\(Reuters\)\s*-\s*", "", text) 
    text = re.sub(r"^WASHINGTON\s*-\s*", "", text)
    return text
# Apply cleaning
true_df['text'] = true_df['text'].apply(clean_text)
fake_df['text'] = fake_df['text'].apply(clean_text)

isot_df = pd.concat([fake_df, true_df]).sample(frac=1, random_state=42).reset_index(drop=True)

isot_df['text_combined'] = isot_df['title'] + " " + isot_df['text']

isot_subset = isot_df.sample(frac=0.05, random_state=42) 

print(f"Training Subset Size: {len(isot_subset)}")

Training Subset Size: 2245


## 4. Tokenization & Dataset Object
Transformers require specific tokenization.

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

X_train, X_val, y_train, y_val = train_test_split(
    isot_subset['text_combined'].tolist(), 
    isot_subset['label'].tolist(), 
    test_size=0.2, 
    random_state=42
)

train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(X_val, truncation=True, padding=True, max_length=128)

train_dataset = NewsDataset(train_encodings, y_train)
val_dataset = NewsDataset(val_encodings, y_val)

## 5. Fine-Tuning DistilBERT
We use the `Trainer` API from Hugging Face.

In [ ]:
# Load Model
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased')


training_args = TrainingArguments(
    output_dir='../Models/BertResults',
    num_train_epochs=1,              
    per_device_train_batch_size=8,   
    per_device_eval_batch_size=16,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir='../Models/BertLogs',
    logging_steps=10,
    eval_strategy="steps",           
    eval_steps=50,
    save_steps=100,
    load_best_model_at_end=True,
    report_to="none"
)

# Metrics function
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {'accuracy': acc, 'f1': f1}

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Starting Training...")
try:
    trainer.train()
    print("Training Complete!")
except Exception as e:
    print(f"Training interrupted: {e}")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting Training...


Step,Training Loss,Validation Loss,Accuracy,F1
50,0.229200,0.198893,0.933185,0.939024
100,0.124000,0.177836,0.957684,0.958606
150,0.054300,0.086858,0.973274,0.974359
200,0.108900,0.069983,0.982183,0.983051


Training Complete!


## 6. Deployment: Save Model
This saves the model in a standard format loadable by cloud services.

In [6]:
save_directory = "../Models/FakeNewsDistilBO"
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)
print(f"Model saved to {save_directory}")

Model saved to ../Models/FakeNewsDistilBO


## 7. Inference Demo (Advanced)
Test the model on custom text.

In [ ]:

import os
import shutil

print("-" * 50)
print("MODEL OPTIMIZATION (Reduce Size)")
print("-" * 50)

optimizer_path = os.path.join(save_directory, "optimizer.pt")
if os.path.exists(optimizer_path):
    os.remove(optimizer_path)
    print(f"Deleted 'optimizer.pt' (Saved ~500MB)")


try:
    print("Applying Dynamic Quantization...")
    quantized_model = torch.quantization.quantize_dynamic(
        model, {torch.nn.Linear}, dtype=torch.qint8
    )
    
    quantized_dir = "../Models/FakeNewsDistilBERT_Quantized"
    if not os.path.exists(quantized_dir):
        os.makedirs(quantized_dir)
        
    torch.save(quantized_model.state_dict(), os.path.join(quantized_dir, "pytorch_model.bin"))
    tokenizer.save_pretrained(quantized_dir)
    print(f"Quantized Model saved to {quantized_dir}")
    
    original_size = os.path.getsize(os.path.join(save_directory, "model.safetensors")) / (1024*1024)
    quant_size = os.path.getsize(os.path.join(quantized_dir, "pytorch_model.bin")) / (1024*1024)
    
    print(f"\nOriginal Size: {original_size:.2f} MB")
    print(f"Quantized Size: {quant_size:.2f} MB")
    print(f"Reduction: {(1 - quant_size/original_size):.2%}")

except Exception as e:
    print(f"Quantization failed: {e}")

model = quantized_model


--------------------------------------------------
MODEL OPTIMIZATION (Reduce Size)
--------------------------------------------------
Applying Dynamic Quantization...
Quantized Model saved to ../Models/FakeNewsDistilBERT_Quantized

Original Size: 255.43 MB
Quantized Size: 132.29 MB
Reduction: 48.21%


In [ ]:
import os
import re
import json
import requests
from datetime import datetime
from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer, DistilBertForSequenceClassification
from retrying import retry
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import html

GOOGLE_FACT_CHECK_KEY = os.getenv("GOOGLE_FACT_CHECK_KEY") 
SERPER_API_KEY = os.getenv("SERPER_API_KEY")

semantic_model = CrossEncoder('cross-encoder/nli-deberta-v3-base')

quantized_dir = "../Models/FakeNewsDistilBERT_Quantized"
tokenizer = AutoTokenizer.from_pretrained(quantized_dir)

model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
model = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8
)
model.load_state_dict(torch.load(os.path.join(quantized_dir, "pytorch_model.bin")))
model.eval()

def extract_smart_query(text):
    noise_patterns = [
        r"breaking news[:\s]*", 
        r"confirmed report[:\s]*", 
        r"sources say[:\s]*",
        r"viral video[:\s]*"
    ]
    clean_text = text
    for p in noise_patterns:
        clean_text = re.sub(p, "", clean_text, flags=re.IGNORECASE)
    return clean_text.strip()

def check_semantic_contradiction(user_claim, evidence_text):
    try:
        results = semantic_model.predict([(user_claim, evidence_text)])
        if isinstance(results, list):
            scores = results[0]
        elif results.ndim > 1:
            scores = results[0]
        else:
            scores = results
        contradiction_score = float(scores[0])
        entailment_score = float(scores[1])
        neutral_score = float(scores[2])
        max_score = max(contradiction_score, entailment_score, neutral_score)
        if max_score == contradiction_score and contradiction_score > 0.6:
            return "CONTRADICT"
        if max_score == entailment_score and entailment_score > 0.6:
            return "SUPPORT"
        return "NEUTRAL"
    except Exception as e:
        return "NEUTRAL"

@retry(stop_max_attempt_number=3, wait_exponential_multiplier=1000)
def check_official_fact_check(query, api_key):
    if not api_key:
        raise ValueError("Missing Google Fact Check API key")
    url = "https://factchecktools.googleapis.com/v1alpha1/claims:search"
    params = {'query': query, 'key': api_key, 'pageSize': 5}
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()
    verdicts = []
    for claim_data in data.get('claims', []):
        review = claim_data.get('claimReview', [{}])[0]
        rating = review.get('textualRating', 'Unknown').lower()
        claim_text = claim_data.get('text', '')
        evidence = f"Fact Check Verdict: The statement '{claim_text}' is rated as {rating}."
        verdict = check_semantic_contradiction(query, evidence)
        if verdict in ["CONTRADICT", "SUPPORT"]:
            publisher = review.get('publisher', {}).get('name', 'Fact Checker')
            verdicts.append((verdict, f"{publisher} rated this claim: {rating}", review.get('url')))
    if verdicts:
        contradict_count = sum(1 for v in verdicts if v[0] == "CONTRADICT")
        support_count = sum(1 for v in verdicts if v[0] == "SUPPORT")
        if contradict_count > support_count:
            return verdicts[0]  
        elif support_count > 0:
            return verdicts[-1]  
    return None

@retry(stop_max_attempt_number=3, wait_exponential_multiplier=1000)
def search_google_serper(query, api_key):
    if not api_key:
        raise ValueError("Missing Serper API key")
    resp = requests.post("https://google.serper.dev/search", 
                         headers={'X-API-KEY': api_key, 'Content-Type': 'application/json'}, 
                         data=json.dumps({"q": query}), timeout=10)
    resp.raise_for_status()
    results = resp.json().get("organic", [])
    verdicts = []
    for res in results[:5]:
        title = res.get('title', '')
        snippet = res.get('snippet', '')
        link = res.get('link', '')
        evidence_chunk = f"{title}. {snippet}"
        verdict = check_semantic_contradiction(query, evidence_chunk)
        if verdict in ["CONTRADICT", "SUPPORT"]:
            verdicts.append((verdict, f"From: {title}", link))
    if verdicts:
        contradict_count = sum(1 for v in verdicts if v[0] == "CONTRADICT")
        support_count = sum(1 for v in verdicts if v[0] == "SUPPORT")
        if contradict_count > support_count:
            return "CONTRADICT", verdicts[0][1], verdicts[0][2]
        elif support_count > contradict_count:
            return "SUPPORT", verdicts[-1][1], verdicts[-1][2]
    return "NEUTRAL", "No consensus", ""

def predict_news_hybrid(text):
    query = extract_smart_query(text)
    
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        output = model(**inputs)
    probs = torch.nn.functional.softmax(output.logits, dim=-1)
    ml_confidence, ml_predicted_class = torch.max(probs, dim=1)
    ml_label = "FAKE" if ml_predicted_class.item() == 1 else "REAL"
    ml_conf = ml_confidence.item()
    
    try:
        official = check_official_fact_check(query, GOOGLE_FACT_CHECK_KEY)
        if official:
            status, reason, link = official
            label = "FAKE" if status == "CONTRADICT" else "REAL"
            conf = 0.99 if ml_label == label else 0.85
            return {"prediction": label, "confidence": conf, "source": "Google Fact Check", "reason": reason, "url": link}
    except Exception:
        pass

    try:
        web_verdict, web_reason, web_link = search_google_serper(query, SERPER_API_KEY)
        if web_verdict != "NEUTRAL":
            label = "FAKE" if web_verdict == "CONTRADICT" else "REAL"
            conf = 0.95 if ml_label == label else 0.75
            return {"prediction": label, "confidence": conf, "source": "Web Search", "reason": web_reason, "url": web_link}
    except Exception:
        pass
    
    return {"prediction": ml_label, "confidence": round(ml_conf, 2), "source": "AI Model", "reason": "Based on text analysis", "url": ""}

def create_ui():
    header = widgets.HTML("<h2>🛡️ AI Fake News Inspector</h2>")
    
    text_input = widgets.Textarea(
        value='', 
        placeholder='Paste claim here...', 
        description='<b>Claim:</b>', 
        layout=widgets.Layout(width='98%', height='100px')
    )
    
    check_button = widgets.Button(
        description='Analyze Claim', 
        button_style='primary', 
        icon='search',
        layout=widgets.Layout(width='200px')
    )
    
    log_output = widgets.Output(layout={'border': '1px solid #ddd', 'height': '200px', 'overflow_y': 'scroll', 'padding': '10px'})
    log_accordion = widgets.Accordion(children=[log_output])
    log_accordion.set_title(0, '📋 Live System Logs (Click to Expand/Collapse)')
    log_accordion.selected_index = 0
    
    result_output = widgets.Output()

    def on_button_click(b):
        with result_output: clear_output()
        with log_output: 
            clear_output()
        user_text = text_input.value.strip()
        if not user_text:
            with result_output: display(HTML("<b style='color:orange;'>⚠️ Please enter text.</b>"))
            return
        try:
            with log_output:
                res = predict_news_hybrid(user_text)
            with result_output:
                colors = {"REAL": "#198754", "FAKE": "#DC3545", "SUSPICIOUS": "#FD7E14", "UNVERIFIED": "#6C757D"}
                bg = colors.get(res.get('prediction'), "#6C757D")
                source = res.get('source', 'Unknown')
                reason = html.escape(res.get('reason', ''))
                url = res.get('url', '')
                display(HTML(f"""
                <div style="border-left: 6px solid {bg}; background: #fff; padding: 20px; box-shadow: 0 2px 5px rgba(0,0,0,0.1); border-radius: 5px;">
                    <h2 style="color: {bg}; margin:0;">{res.get('prediction')} <small style="color:#555; font-size:0.6em;">({res.get('confidence'):.0%} Confidence)</small></h2>
                    <hr style="margin: 10px 0; opacity: 0.2;">
                    <p><b>🔍 Source:</b> {source}</p>
                    <p><b>📝 Reason:</b> <i>"{reason}"</i></p>
                    {f'<p><b>🔗 Proof:</b> <a href="{url}" target="_blank">{url}</a></p>' if url else ''}
                </div>
                """))
        except Exception as e:
            with log_output:
                print(f"\n❌ CRITICAL ERROR: {str(e)}")
            with result_output:
                display(HTML(f"<b style='color:red;'>❌ Error. Check the logs above for details.</b>"))

    check_button.on_click(on_button_click)
    
    display(widgets.VBox([header, text_input, check_button, log_accordion, result_output]))

create_ui()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\yousef\AppData\Local\Temp\ipykernel_5772\4263221785.py:27: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless th